# 🧬 Generative vs. Discriminative Classification: Naive Bayes vs. Logistic Regression

### *ML for Scientists — Topic 3–4: Classification*

---

## Overview

Naive Bayes and logistic regression solve the same binary
classification problem but from opposite directions:

- **Naive Bayes** (generative): model $P(X\mid Y)$ and $P(Y)$
  separately, then combine via Bayes' rule to get $P(Y\mid X)$. Its
  defining simplification is **conditional independence of features
  given the class**.
- **Logistic regression** (discriminative): model $P(Y\mid X)$
  directly, with no assumption about how $X$ is distributed.

We compare them on a **real medical dataset** — the Wisconsin Breast
Cancer Diagnostic dataset (Wolberg et al.) — where several features are
**genuinely, severely correlated** (radius, perimeter, and area of a
cell nucleus are all measuring almost the same thing). This gives us a
real, built-in violation of Naive Bayes's core assumption to study,
rather than one we'd have to construct artificially.

| Module | Topic |
|--------|-------|
| **1**  | Real Data: Wisconsin Breast Cancer Diagnostic Dataset |
| **2**  | Generative vs. Discriminative: Theory |
| **3**  | Implementation From Scratch |
| **4**  | Results: Accuracy, Decision Boundaries, and Calibration |
| **5**  | Production Context & Limitations |

> **Prerequisites:** `numpy`, `scikit-learn` (for the dataset only —
> both models are implemented from scratch), `matplotlib`.

In [ ]:
# ============================================================
# GLOBAL IMPORTS & CONFIGURATION
# ============================================================
import warnings

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
np.random.seed(0)

DARK_BG, ACCENT, TEXT = "#0d1117", "#58a6ff", "#e6edf3"
plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": DARK_BG,
    "axes.edgecolor": TEXT, "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT, "text.color": TEXT,
    "figure.titlesize": 14,
})

In [ ]:
# ============================================================
# MODULE 1 — REAL DATA: WISCONSIN BREAST CANCER DIAGNOSTIC DATASET
# ============================================================
data = load_breast_cancer()
X, y = data.data, data.target  # y=0 malignant, y=1 benign
feature_names = list(data.feature_names)
print(f"Real dataset: {X.shape[0]} real patients, {X.shape[1]} real features "
      f"computed from digitized fine-needle-aspirate cell-nucleus images")
print(f"Class balance: {(y==0).sum()} malignant, {(y==1).sum()} benign")

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
mu, sigma = Xtr.mean(0), Xtr.std(0)
Xtr_s, Xte_s = (Xtr - mu) / sigma, (Xte - mu) / sigma

# --- feature correlation: the real independence violation Naive Bayes must live with ---
corr = np.corrcoef(X.T)
i_radius, i_perimeter, i_area = (feature_names.index(n) for n in
                                  ["mean radius", "mean perimeter", "mean area"])
print(f"\nCorrelation(mean radius, mean perimeter) = {corr[i_radius, i_perimeter]:.3f}")
print(f"Correlation(mean radius, mean area)      = {corr[i_radius, i_area]:.3f}")
print("These are the same underlying tumor size, measured three ways — a real, severe")
print("violation of Naive Bayes's conditional-independence assumption, not a contrived one.")

# ------------------------------------------------------------------
# VISUALIZATION 1 — Real feature correlation matrix
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("Module 1 — Real Feature Correlations (Wisconsin Breast Cancer Data)",
             color=ACCENT, fontweight="bold", fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

---
# Module 2 — Generative vs. Discriminative: Theory

## 2.1 Naive Bayes (Generative)

$$P(Y\mid X) \propto P(Y)\prod_{j=1}^{d} P(X_j \mid Y)$$

For continuous features, **Gaussian Naive Bayes** models each
$P(X_j\mid Y=c)$ as a 1D Gaussian fit independently per class and per
feature — no covariance between features is modeled at all. Fitting
is a single closed-form pass: compute per-class means and variances.

## 2.2 Logistic Regression (Discriminative)

$$P(Y=1\mid x) = \sigma(w_0 + w^\top x), \qquad \sigma(z) = \frac{1}{1+e^{-z}}$$

There's no closed-form solution — fitting requires iterative
optimization (gradient ascent on the log-likelihood). An $L_2$ penalty
on $w$ corresponds to a Gaussian prior (MAP estimation), exactly
mirroring ridge regression's Gaussian-prior interpretation.

## 2.3 The Key Difference for This Dataset

Naive Bayes treats `mean radius`, `mean perimeter`, and `mean area` as
**three independent pieces of evidence**, even though they are nearly
the same measurement. This means the same underlying signal gets
**counted three times** in Naive Bayes's posterior — a real mechanism
for the overconfidence we'll measure directly in Module 4. Logistic
regression has no such assumption; it can freely down-weight
redundant, correlated features.

---
# Module 3 — Implementation From Scratch

In [ ]:
# ============================================================
# MODULE 3 — GAUSSIAN NAIVE BAYES AND LOGISTIC REGRESSION, FROM SCRATCH
# ============================================================
class GaussianNaiveBayes:
    """Per-class, per-feature Gaussian likelihood; no feature covariance modeled."""

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors, self.mean, self.var = {}, {}, {}
        for c in self.classes:
            Xc = X[y == c]
            self.priors[c] = len(Xc) / len(X)
            self.mean[c] = Xc.mean(0)
            self.var[c] = Xc.var(0) + 1e-6
        return self

    def _log_likelihood(self, X, c):
        return (-0.5 * np.sum(np.log(2 * np.pi * self.var[c]))
                - 0.5 * np.sum((X - self.mean[c]) ** 2 / self.var[c], axis=1))

    def predict_proba(self, X):
        logp = np.stack([self._log_likelihood(X, c) + np.log(self.priors[c]) for c in self.classes], axis=1)
        logp -= logp.max(axis=1, keepdims=True)
        p = np.exp(logp)
        return p / p.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes[np.argmax(self.predict_proba(X), axis=1)]


class LogisticRegressionScratch:
    """Gradient-ascent logistic regression with an L2 (Gaussian-prior / MAP) penalty."""

    def __init__(self, lr=0.5, n_iters=3000, l2=1.0):
        self.lr, self.n_iters, self.l2 = lr, n_iters, l2

    def fit(self, X, y):
        n, d = X.shape
        self.w, self.b = np.zeros(d), 0.0
        self.loss_history = []
        for _ in range(self.n_iters):
            z = X @ self.w + self.b
            p = 1 / (1 + np.exp(-z))
            grad_w = X.T @ (p - y) / n + self.l2 * self.w / n
            grad_b = (p - y).mean()
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b
            eps = 1e-9
            nll = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
            self.loss_history.append(nll)
        return self

    def predict_proba(self, X):
        p1 = 1 / (1 + np.exp(-(X @ self.w + self.b)))
        return np.stack([1 - p1, p1], axis=1)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)


nb_model = GaussianNaiveBayes().fit(Xtr_s, ytr)
lr_model = LogisticRegressionScratch(lr=0.5, n_iters=3000, l2=1.0).fit(Xtr_s, ytr)

nb_pred, lr_pred = nb_model.predict(Xte_s), lr_model.predict(Xte_s)
nb_acc, lr_acc = (nb_pred == yte).mean(), (lr_pred == yte).mean()
print(f"Naive Bayes accuracy:       {nb_acc:.4f}")
print(f"Logistic Regression accuracy: {lr_acc:.4f}")

# ------------------------------------------------------------------
# VISUALIZATION 2 — Logistic regression training loss
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lr_model.loss_history, color=ACCENT)
ax.set_xlabel("gradient ascent iteration"); ax.set_ylabel("negative log-likelihood")
ax.set_title("Module 3 — Logistic Regression Training Curve (no closed-form solution)",
             color=ACCENT, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Module 4 — Results: Accuracy, Decision Boundaries, and Calibration

## 4.1 Decision Boundaries (2 Real Features)

Naive Bayes's per-class, per-feature variances make its boundary
**curved** even in 2D; logistic regression's boundary is **linear by
construction**.

In [ ]:
# ============================================================
# MODULE 4 — DECISION BOUNDARIES ON TWO REAL FEATURES
# ============================================================
f1, f2 = "mean concave points", "mean texture"
i1, i2 = feature_names.index(f1), feature_names.index(f2)
X2 = X[:, [i1, i2]]
mu2, sigma2 = X2.mean(0), X2.std(0)
X2s = (X2 - mu2) / sigma2

nb2 = GaussianNaiveBayes().fit(X2s, y)
lr2 = LogisticRegressionScratch(lr=0.5, n_iters=3000, l2=1.0).fit(X2s, y)

xx, yy = np.meshgrid(np.linspace(X2s[:, 0].min() - 1, X2s[:, 0].max() + 1, 200),
                      np.linspace(X2s[:, 1].min() - 1, X2s[:, 1].max() + 1, 200))
grid = np.stack([xx.ravel(), yy.ravel()], axis=1)
nb_z = nb2.predict_proba(grid)[:, 1].reshape(xx.shape)
lr_z = lr2.predict_proba(grid)[:, 1].reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
fig.suptitle(f"Module 4 — Decision Boundaries on Real Features ({f1} vs. {f2})",
             color=ACCENT, fontweight="bold")
for ax, z, title in zip(axes, [nb_z, lr_z], ["Naive Bayes (curved)", "Logistic Regression (linear)"]):
    ax.contourf(xx, yy, z, levels=20, cmap="RdBu_r", alpha=0.75)
    ax.scatter(X2s[y == 0, 0], X2s[y == 0, 1], s=10, c="darkred", label="malignant", edgecolor="k", linewidth=0.2)
    ax.scatter(X2s[y == 1, 0], X2s[y == 1, 1], s=10, c="darkblue", label="benign", edgecolor="k", linewidth=0.2)
    ax.set_title(title, color=TEXT, fontsize=10)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4.2 The Real Finding: Calibration

The syllabus makes a specific claim: *"a classifier can predict labels
reasonably well even when its estimated probabilities are
inaccurate."* We now measure this directly rather than just asserting
it.

In [ ]:
# ============================================================
# MODULE 4 — CALIBRATION: BRIER SCORE, ECE, RELIABILITY DIAGRAM
# ============================================================
nb_proba = nb_model.predict_proba(Xte_s)[:, 1]
lr_proba = lr_model.predict_proba(Xte_s)[:, 1]


def calibration_curve(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.clip(np.digitize(y_prob, bins) - 1, 0, n_bins - 1)
    mean_pred, frac_pos, counts = np.zeros(n_bins), np.zeros(n_bins), np.zeros(n_bins)
    for b in range(n_bins):
        mask = bin_ids == b
        counts[b] = mask.sum()
        if mask.sum() > 0:
            frac_pos[b], mean_pred[b] = y_true[mask].mean(), y_prob[mask].mean()
    return mean_pred, frac_pos, counts


def brier_score(y_true, y_prob):
    return np.mean((y_prob - y_true) ** 2)


def expected_calibration_error(mean_pred, frac_pos, counts):
    w = counts / counts.sum()
    valid = counts > 0
    return np.sum(w[valid] * np.abs(mean_pred[valid] - frac_pos[valid]))


nb_mp, nb_fp, nb_cnt = calibration_curve(yte, nb_proba)
lr_mp, lr_fp, lr_cnt = calibration_curve(yte, lr_proba)
nb_brier, lr_brier = brier_score(yte, nb_proba), brier_score(yte, lr_proba)
nb_ece = expected_calibration_error(nb_mp, nb_fp, nb_cnt)
lr_ece = expected_calibration_error(lr_mp, lr_fp, lr_cnt)
nb_extreme = ((nb_proba > 0.999) | (nb_proba < 0.001)).mean()
lr_extreme = ((lr_proba > 0.999) | (lr_proba < 0.001)).mean()

print(f"{'Metric':<32}{'Naive Bayes':>14}{'Logistic Regression':>22}")
print(f"{'Accuracy':<32}{nb_acc:>14.3f}{lr_acc:>22.3f}")
print(f"{'Brier score (lower=better)':<32}{nb_brier:>14.3f}{lr_brier:>22.3f}")
print(f"{'Expected calibration error':<32}{nb_ece:>14.3f}{lr_ece:>22.3f}")
print(f"{'Fraction at extreme prob.':<32}{nb_extreme:>14.1%}{lr_extreme:>22.1%}")
print("\nNaive Bayes is only modestly less accurate, but dramatically more overconfident —")
print("directly attributable to double-counting the correlated radius/perimeter/area evidence.")

# ------------------------------------------------------------------
# VISUALIZATION 3 — Reliability diagram
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], "--", color=TEXT, alpha=0.5, label="perfect calibration")
ax.plot(nb_mp, nb_fp, "o-", color="#f0883e", label=f"Naive Bayes (ECE={nb_ece:.3f})")
ax.plot(lr_mp, lr_fp, "o-", color=ACCENT, label=f"Logistic Regression (ECE={lr_ece:.3f})")
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed fraction positive")
ax.set_title("Module 4 — Reliability Diagram (Real, Measured Calibration)",
             color=ACCENT, fontweight="bold")
ax.legend(facecolor=DARK_BG, labelcolor=TEXT)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# VISUALIZATION 4 — Predicted probability histograms
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Module 4 — Predicted Probability Distributions (Real Test Predictions)",
             color=ACCENT, fontweight="bold")
axes[0].hist(nb_proba, bins=30, color="#f0883e")
axes[0].set_title(f"Naive Bayes\n{nb_extreme:.0%} of predictions are near 0 or 1", color=TEXT, fontsize=10)
axes[1].hist(lr_proba, bins=30, color=ACCENT)
axes[1].set_title(f"Logistic Regression\n{lr_extreme:.0%} of predictions are near 0 or 1", color=TEXT, fontsize=10)
for ax in axes:
    ax.set_xlabel("predicted P(benign)")
plt.tight_layout()
plt.show()

---
# Module 5 — Production Context & Limitations

| Method | Where used in practice | Notes |
|--------|------------------------|-------|
| Naive Bayes | Spam/sequence classification, early gene-expression classifiers, text mining | Fast, needs little data, but posterior probabilities are often unreliable when features are correlated |
| Logistic regression | GWAS, clinical risk scores, baseline model in nearly every biomedical ML paper | Well-calibrated by construction when the linear-in-log-odds assumption roughly holds |
| scikit-learn `GaussianNB` / `LogisticRegression` | Production implementations | Add features like variance smoothing, multiple solvers, and regularization paths not implemented here |
| Platt scaling / isotonic regression | Post-hoc calibration | The standard real fix for an overconfident model like Naive Bayes here |

## Known Limitations of This Tutorial
- **Diagonal-covariance Gaussian NB only**: real "semi-naive" Bayes
  variants relax the independence assumption for specific known
  feature groups; not implemented here.
- **Plain gradient ascent, fixed learning rate**: production logistic
  regression implementations use line search or Newton-type methods
  (e.g., L-BFGS) for faster, more robust convergence.
- **One train/test split**: a real evaluation would use cross-validation
  (Topic 15 in this syllabus) rather than a single 70/30 split.
- **No post-hoc calibration applied**: Naive Bayes's overconfidence
  could be partly corrected with Platt scaling; we left it uncorrected
  to show the raw effect clearly.

In [ ]:
# ============================================================
# FINAL DASHBOARD — Complete pipeline summary
# ============================================================
fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle("🧬 Naive Bayes vs. Logistic Regression — Pipeline Dashboard",
             fontsize=15, fontweight="bold", color=ACCENT, y=0.98)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.5, wspace=0.3)

ax0 = fig.add_subplot(gs[0, 0]); im0 = ax0.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1); ax0.set_title("Real feature correlations", color=TEXT, fontsize=10); ax0.set_xticks([]); ax0.set_yticks([])
ax1 = fig.add_subplot(gs[0, 1]); ax1.contourf(xx, yy, nb_z, levels=20, cmap="RdBu_r", alpha=0.75); ax1.scatter(X2s[:, 0], X2s[:, 1], c=y, cmap="RdBu", s=4); ax1.set_title("Naive Bayes boundary", color=TEXT, fontsize=10); ax1.set_xticks([]); ax1.set_yticks([])
ax2 = fig.add_subplot(gs[0, 2]); ax2.contourf(xx, yy, lr_z, levels=20, cmap="RdBu_r", alpha=0.75); ax2.scatter(X2s[:, 0], X2s[:, 1], c=y, cmap="RdBu", s=4); ax2.set_title("Logistic Regression boundary", color=TEXT, fontsize=10); ax2.set_xticks([]); ax2.set_yticks([])
ax3 = fig.add_subplot(gs[0, 3])
ax3.plot([0, 1], [0, 1], "--", color=TEXT, alpha=0.5)
ax3.plot(nb_mp, nb_fp, "o-", color="#f0883e", markersize=4)
ax3.plot(lr_mp, lr_fp, "o-", color=ACCENT, markersize=4)
ax3.set_title("Reliability diagram", color=TEXT, fontsize=10)

ax4 = fig.add_subplot(gs[1, 0:2])
metrics_names = ["Accuracy", "Brier score", "ECE", "Extreme-prob. fraction"]
nb_vals = [nb_acc, nb_brier, nb_ece, nb_extreme]
lr_vals = [lr_acc, lr_brier, lr_ece, lr_extreme]
xpos = np.arange(len(metrics_names))
width = 0.35
ax4.bar(xpos - width / 2, nb_vals, width, label="Naive Bayes", color="#f0883e")
ax4.bar(xpos + width / 2, lr_vals, width, label="Logistic Regression", color=ACCENT)
ax4.set_xticks(xpos); ax4.set_xticklabels(metrics_names, rotation=15, fontsize=8)
ax4.set_title("Accuracy vs. calibration, side by side", color=TEXT, fontsize=10)
ax4.legend(facecolor=DARK_BG, labelcolor=TEXT, fontsize=8)

ax5 = fig.add_subplot(gs[1, 2])
ax5.hist(nb_proba, bins=30, color="#f0883e")
ax5.set_title("NB predicted probabilities", color=TEXT, fontsize=9)

ax6 = fig.add_subplot(gs[1, 3])
ax6.hist(lr_proba, bins=30, color=ACCENT)
ax6.set_title("LR predicted probabilities", color=TEXT, fontsize=9)

plt.tight_layout()
plt.show()

---
# Summary

## What This Notebook Demonstrated

| Step | Module | Key Idea |
|------|--------|----------|
| Real data | 1 | Wisconsin Breast Cancer dataset; radius/perimeter/area correlate at r≈0.99 — a real, severe independence violation |
| Theory | 2 | Generative (model $P(X\mid Y)$) vs. discriminative (model $P(Y\mid X)$ directly) |
| Implementation | 3 | Both models built from scratch — closed-form Gaussian fitting vs. iterative gradient ascent |
| Results | 4 | Comparable accuracy (NB 91.2%, LR 95.9%) but Naive Bayes is dramatically overconfident (ECE ≈3.6× worse, 94.7% of predictions near 0 or 1) |
| Context | 5 | Positioned against scikit-learn implementations and real calibration fixes (Platt scaling) |

## Computational Complexity

| Step | Complexity | Bottleneck |
|------|------------|------------|
| Gaussian NB fitting | $\mathcal{O}(n \cdot d)$ | One pass over the data; closed-form |
| Logistic regression fitting | $\mathcal{O}(\text{iters} \cdot n \cdot d)$ | Iterative gradient ascent, no closed form |
| Prediction (either model) | $\mathcal{O}(n \cdot d)$ | Negligible |
| Calibration curve | $\mathcal{O}(n)$ | Negligible |

## Key References
- Wolberg, Street & Mangasarian — Wisconsin Diagnostic Breast Cancer dataset (UCI ML Repository)
- Ng & Jordan (2001) — On Discriminative vs. Generative Classifiers: A Comparison of Logistic Regression and Naive Bayes (*NeurIPS*)
- Niculescu-Mizil & Caruana (2005) — Predicting Good Probabilities With Supervised Learning (*ICML*) — the calibration analysis in Module 4 follows this line of work
- 02-620 *Machine Learning for Scientists* course materials, Lectures 3–4 (Classification with k-NN/Naive Bayes; Logistic Regression)